In [1]:
# This cell is removed with the tag: "remove-input"
# As such, it will not be shown in documentation

import warnings
warnings.filterwarnings('ignore')


(Tutorial_Copy)=
# Copy

*Creating independent copies of molecular systems.*

The function {func}`molsysmt.basic.copy` allows you to create a fully independent copy of a molecular system.

:::{versionadded} 1.0.0
:::

:::{admonition} API documentation
:class: dropdown

Follow this link for a detailed description of the input arguments, raised errors, and returned objects of this function: {func}`molsysmt.basic.copy`.
:::

For standalone native `Structures`, `structures.copy()` eagerly copies coordinate/box bytes, while `structures.copy(deep=False)` shares protected blocks until edits replace the affected blocks. Both modes own independent metadata and other mutable series. Python `copy.copy(structures)` selects deferred geometry allocation; `copy.deepcopy(structures)` selects eager allocation. Native `molsys.copy(deep=False)` and Python `copy.copy(molsys)` apply protected geometry sharing to the complete system, with independent canonical tables, domains and named analyses. `molsys.copy()`, Python `copy.deepcopy(molsys)` and universal `msm.copy(molsys)` copy geometry eagerly. Copying retains index spaces, occurrence identities, producer versions and missing domains without remapping or reconstruction. Edit coordinates through `msm.set` to invalidate only the copy’s affected structures. Direct attached-domain edits still require explicit invalidation. See {ref}`MolSys copies <user-foundations-molsys-copying>` and {ref}`Structures copies <user-foundations-structures-copying>`. Native `Topology`, `ChemicalStates` and `MolecularMechanics` copies detach canonical tables eagerly. `MolSysBuilder.copy(deep=False)` preserves incomplete staging over protected geometry without compiling it. Native payload wrappers also own their mutable data under both Python copy operations; plain dictionary forms retain Python shallow-copy semantics when used directly. The universal `msm.copy` returns an independent registered-form snapshot. See {ref}`Native Copying <user-foundations-resident-copying>`.


## Basic usage

Let's show how this function works using a PDB structure (`181L`):


In [2]:
import molsysmt as msm

In [3]:
molsys = msm.convert(msm.systems['T4 lysozyme L99A']['181l.h5msm'], to_form='molsysmt.MolSys')

:::{note}
:class: dropdown

**Demo Systems Catalog**

This tutorial uses the bundled 181L system. See the
{ref}`Demo Systems <user-foundations-entrance-demo-systems>` catalog.
:::

In [4]:
msm.info(molsys)

form,n_atoms,n_groups,n_components,n_chains,n_molecules,n_entities,n_waters,n_ions,n_small_molecules,n_proteins,n_structures
molsysmt.MolSys,1441,302,141,6,141,5,136,2,2,1,1


In [5]:
molsys_copy = msm.copy(molsys)

:::{tip}
:class: dropdown

All methods defined in the {ref}`molsysmt.basic <API basic>` module can also be invoked from the main level of the library. Hence, {func}`molsysmt.copy` is the same function as {func}`molsysmt.basic.copy`.
:::

Although the original and copied systems are structurally identical, they are stored as distinct Python objects:

In [6]:
id(molsys)!= id(molsys_copy)

True

But they represent the same molecular system:

In [7]:
msm.compare(molsys, molsys_copy, rule='equal')

True

In [8]:
msm.info(molsys_copy)

form,n_atoms,n_groups,n_components,n_chains,n_molecules,n_entities,n_waters,n_ions,n_small_molecules,n_proteins,n_structures
molsysmt.MolSys,1441,302,141,6,141,5,136,2,2,1,1


## Copying a file

Without a filename, MolSysMT creates an independent temporary in a private
system-temporary directory. It remains usable through the current Python session;
normal-exit cleanup is best effort. An explicit destination creates a persistent
caller-owned copy. Both retain the stored bytes and format version, including
legacy H5MSM versions. Source aliases and directories reject with `ArgumentError`.
Composite sources receive individual member copies. Copying does not add missing
format write capabilities. See {ref}`Cookbook_Copying_Disk_Systems`.


In [9]:
from pathlib import Path
from tempfile import TemporaryDirectory
import filecmp

with TemporaryDirectory() as directory:
    source = msm.convert(molsys, to_form=Path(directory) / 'source.h5msm')
    temporary = msm.copy(source)
    snapshot = msm.copy(source, output_filename=Path(directory) / 'snapshot.h5msm')
    assert filecmp.cmp(source, temporary, shallow=False)
    assert filecmp.cmp(source, snapshot, shallow=False)
    print("Independent snapshot:", Path(snapshot).name, msm.get_form(snapshot))
assert Path(temporary).is_file()  # its lifetime belongs to the Python session


Independent snapshot: snapshot.h5msm file:h5msm


:::{seealso}
:class: dropdown

**Related Tools & References**

- {ref}`Tutorial_Convert`: Convert a molecular system into another form or forms with {func}`molsysmt.basic.convert`.
- {ref}`Tutorial_Info`: Print a summary of the contents, topology, and structural data of a molecular system with {func}`molsysmt.basic.info`.
- {ref}`Tutorial_Compare`: Compare molecular systems across selected attributes with {func}`molsysmt.basic.compare`.
:::